In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
rec = pd.read_csv(r"C:\Users\naren\Desktop\Projects\TripNova\Backend\Dataset\Recommendation.csv")

print("Dataset shape:", rec.shape)
rec.head()

Dataset shape: (172, 74)


,place_id,place_name,category,subcategory,area,description,estimated_cost_inr,rating_seed,review_count_seed,duration_hours,...,food_weight,photography_weight,relaxation_weight,latitude_seed,longitude_seed,distance_from_chennai_central_km_seed,distance_from_airport_km_seed,data_source_type,verification_status,synthetic_note
0,TN0001,Marina Beach,Beach,Urban Beach,Marina,Marina Beach is a well-known urban beach locat...,233,4.6,417,1.6,...,0.0,0.72,0.68,13.05382,80.28271,3.3,13.8,real_landmark_synthetic_attributes,needs_real_world_verification,Dataset contains synthetic/curated seed values...
1,TN0002,Besant Nagar Beach (Elliot's Beach),Beach,Urban Beach,Besant Nagar,Besant Nagar Beach (Elliot's Beach) is a well-...,241,4.0,560,2.5,...,0.0,0.56,0.88,12.99891,80.27018,9.3,10.8,real_landmark_synthetic_attributes,needs_real_world_verification,Dataset contains synthetic/curated seed values...
2,TN0003,Thiruvanmiyur Beach,Beach,Neighbourhood Beach,Thiruvanmiyur,Thiruvanmiyur Beach is a well-known neighbourh...,285,4.2,144,2.4,...,0.0,0.71,0.80,12.98627,80.25647,10.9,9.3,real_landmark_synthetic_attributes,needs_real_world_verification,Dataset contains synthetic/curated seed values...
3,TN0004,Palavakkam Beach,Beach,Neighbourhood Beach,Palavakkam,Palavakkam Beach is a well-known neighbourhood...,147,4.0,67,1.8,...,0.0,0.65,0.77,12.95733,80.25480,14.1,10.0,real_landmark_synthetic_attributes,needs_real_world_verification,Dataset contains synthetic/curated seed values...
4,TN0005,Kottivakkam Beach,Beach,Neighbourhood Beach,Kottivakkam,Kottivakkam Beach is a well-known neighbourhoo...,105,3.8,2869,2.2,...,0.0,0.81,0.65,12.96877,80.25815,12.8,9.9,real_landmark_synthetic_attributes,needs_real_world_verification,Dataset contains synthetic/curated seed values...


In [3]:
print("Duplicate rows:", rec.duplicated().sum())
print("Total missing values:", rec.isna().sum().sum())

Duplicate rows: 0
Total missing values: 0


In [4]:
destination_info = rec[
    [
        "place_id",
        "place_name",
        "area",
        "description",
        "rating_seed",
        "review_count_seed"
    ]
].copy()

print("Destination data shape:", destination_info.shape)
destination_info.head()

Destination data shape: (172, 6)


,place_id,place_name,area,description,rating_seed,review_count_seed
0,TN0001,Marina Beach,Marina,Marina Beach is a well-known urban beach locat...,4.6,417
1,TN0002,Besant Nagar Beach (Elliot's Beach),Besant Nagar,Besant Nagar Beach (Elliot's Beach) is a well-...,4.0,560
2,TN0003,Thiruvanmiyur Beach,Thiruvanmiyur,Thiruvanmiyur Beach is a well-known neighbourh...,4.2,144
3,TN0004,Palavakkam Beach,Palavakkam,Palavakkam Beach is a well-known neighbourhood...,4.0,67
4,TN0005,Kottivakkam Beach,Kottivakkam,Kottivakkam Beach is a well-known neighbourhoo...,3.8,2869


In [5]:
rec_data = rec.drop(
    [
        "place_id",
        "description",
        "walking_required",
        "vehicle_recommended",
        "elderly_friendly",
        "child_friendly",
        "wheelchair_friendly",
        "latitude_seed",
        "longitude_seed",
        "data_source_type",
        "verification_status",
        "synthetic_note",
        "weekday",
        "weekend",
        "budget_friendly",
        "premium"
    ],
    axis=1
).copy()


In [6]:
travel_group_cols = [
    "family",
    "friends",
    "couple",
    "solo"
]

time_cols = [
    "morning",
    "afternoon",
    "evening",
    "night"
]

interest_cols = [
    "nature",
    "beach",
    "adventure",
    "history",
    "religious",
    "culture",
    "shopping",
    "food",
    "photography",
    "relaxation",
    "wildlife",
    "art",
    "science",
    "entertainment",
    "heritage",
    "indoor",
    "outdoor",
    "romantic",
    "peaceful",
    "crowded",
    "quiet",
    "educational",
    "fun",
    "scenic",
    "instagrammable"
]

interest_weight_cols = [
    "nature_weight",
    "beach_weight",
    "adventure_weight",
    "history_weight",
    "religious_weight",
    "culture_weight",
    "shopping_weight",
    "food_weight",
    "photography_weight",
    "relaxation_weight"
]

duration_cols = [
    "duration_hours",
    "short_visit",
    "half_day",
    "full_day"
]

## 1. Scoring functions

In [7]:
def budget_score(user_budget, destination_budget):
    budget_levels = {
        "Free": 0,
        "Budget": 1,
        "Moderate": 2,
        "High": 3,
        "Premium": 4
    }

    user_budget = str(user_budget).strip().title()
    destination_budget = str(destination_budget).strip().title()

    if user_budget not in budget_levels or destination_budget not in budget_levels:
        return 0.0

    difference = abs(
        budget_levels[user_budget] -
        budget_levels[destination_budget]
    )

    if difference == 0:
        return 1.0
    elif difference == 1:
        return 0.7
    elif difference == 2:
        return 0.4
    else:
        return 0.0

In [8]:
def time_score(destination, user_time):
    column = str(user_time).strip().lower()

    if column not in time_cols:
        return 0.0

    return float(destination[column])

In [9]:
def interest_score(destination, user_interests):
    if not user_interests:
        return None

    scores = []

    for interest in user_interests:
        column = str(interest).strip().lower()

        if not column:
            continue

        weight_column = f"{column}_weight"

        if weight_column in destination.index:
            scores.append(float(destination[weight_column]))
        elif column in interest_cols:
            scores.append(float(destination[column]))

    if not scores:
        return None

    return float(np.mean(scores))

In [10]:
def duration_score(destination, user_duration):
    target_hours = {
        "Short Visit": 2,
        "Half Day": 4,
        "Full Day": 8
    }

    user_duration = str(user_duration).strip().title()

    if user_duration not in target_hours:
        return None

    target = target_hours[user_duration]
    actual = float(destination["duration_hours"])

    difference = abs(actual - target)

    return max(0.0, min(1.0, 1 - (difference / target)))

In [11]:
def rating_score(destination):
    return max(0.0, min(1.0, float(destination["rating_seed"]) / 5))

In [12]:
def travel_group_score(destination, user_group):
    column = str(user_group).strip().lower()

    if column not in travel_group_cols:
        return None

    return float(destination[column])


def travel_feature_available(candidates, user_group):
    column = str(user_group).strip().lower()

    if column not in travel_group_cols:
        return False

    return candidates[column].nunique(dropna=True) > 1

## 2. Final scoring

In [13]:
WEIGHTS = {
    "travel": 0.20,
    "budget": 0.20,
    "time": 0.15,
    "duration": 0.15,
    "interest": 0.25,
    "rating": 0.05
}

In [14]:
def calculate_score(
    destination,
    travelling_with=None,
    budget=None,
    preferred_time=None,
    duration=None,
    interests=None,
    use_travel=True
):
    scores = {}
    weights = {}

    # Travelling With
    if travelling_with and use_travel:
        score = travel_group_score(destination, travelling_with)

        if score is not None:
            scores["travel"] = score
            weights["travel"] = WEIGHTS["travel"]

    # Budget
    if budget:
        scores["budget"] = budget_score(
            budget,
            destination["budget_level"]
        )
        weights["budget"] = WEIGHTS["budget"]

    # Preferred Time
    if preferred_time:
        scores["time"] = time_score(
            destination,
            preferred_time
        )
        weights["time"] = WEIGHTS["time"]

    # Duration
    if duration:
        score = duration_score(destination, duration)

        if score is not None:
            scores["duration"] = score
            weights["duration"] = WEIGHTS["duration"]

    # Interests
    if interests:
        score = interest_score(destination, interests)

        if score is not None:
            scores["interest"] = score
            weights["interest"] = WEIGHTS["interest"]

    # Rating
    if "rating_seed" in destination.index:
        scores["rating"] = rating_score(destination)
        weights["rating"] = WEIGHTS["rating"]

    if not scores:
        return 0.0

    total_weight = sum(weights.values())

    final_score = sum(
        scores[feature] * weights[feature]
        for feature in scores
    ) / total_weight

    return round(final_score * 100, 2)

## 3. Recommendation function

In [15]:
def recommend(
    category,
    travelling_with=None,
    budget=None,
    preferred_time=None,
    duration=None,
    interests=None,
    top_n=10
):
    category = str(category).strip()

    candidates = rec_data[
        rec_data["category"].str.strip().str.lower() == category.lower()
    ].copy()

    if candidates.empty:
        print(f"No destinations found for category: {category}")
        return pd.DataFrame()

    # Use Travelling With only when it varies inside this category.
    use_travel = (
        travelling_with is not None
        and travel_feature_available(candidates, travelling_with)
    )

    candidates["score"] = candidates.apply(
        lambda row: calculate_score(
            row,
            travelling_with=travelling_with,
            budget=budget,
            preferred_time=preferred_time,
            duration=duration,
            interests=interests,
            use_travel=use_travel
        ),
        axis=1
    )

    candidates = candidates.sort_values(
        by=["score", "rating_seed"],
        ascending=[False, False]
    ).copy()

    return candidates.head(top_n)

## 4. Test the recommendation model

In [16]:
results = recommend(
    category="Nature & Parks",
    travelling_with="Friends",
    budget="Moderate",
    preferred_time="Afternoon",
    duration="Half Day",
    interests=["Photography", "Adventure"],
    top_n=10
)

print(
    results[
        [
            "place_name",
            "category",
            "subcategory",
            "budget_level",
            "rating_seed",
            "score"
        ]
    ].to_string(index=False)
)

                            place_name       category        subcategory budget_level  rating_seed  score
              Perumbakkam Wetland Park Nature & Parks       Wetland Park       Budget          4.2  63.84
Chetpet Eco Park Family Picnic Session Nature & Parks Curated Experience       Budget          3.5  63.75
                          Panagal Park Nature & Parks Neighbourhood Park       Budget          4.5  61.09
                    Nageswara Rao Park Nature & Parks Neighbourhood Park       Budget          4.0  60.16
 Semmozhi Poonga Butterfly Garden Walk Nature & Parks Curated Experience       Budget          3.9  59.87
            Nanmangalam Reserve Forest Nature & Parks     Reserve Forest       Budget          4.0  59.69
       Guindy Park Morning Nature Walk Nature & Parks Curated Experience       Budget          4.6  58.72
                  Guindy National Park Nature & Parks      National Park       Budget          4.3  58.50
                 Anna Nagar Tower Park Nature 

## 5. Test when the user does not know their interests

In [17]:
results_no_interests = recommend(
    category="Nature & Parks",
    travelling_with="Friends",
    budget="Moderate",
    preferred_time="Afternoon",
    duration="Half Day",
    interests=None,
    top_n=10
)

print(
    results_no_interests[
        ["place_name", "subcategory", "score"]
    ].to_string(index=False)
)

                            place_name        subcategory  score
                          Panagal Park Neighbourhood Park  80.68
            Nanmangalam Reserve Forest     Reserve Forest  79.09
              Perumbakkam Wetland Park       Wetland Park  78.77
                    Nageswara Rao Park Neighbourhood Park  78.41
Chetpet Eco Park Family Picnic Session Curated Experience  78.18
 Semmozhi Poonga Butterfly Garden Walk Curated Experience  76.18
       Guindy Park Morning Nature Walk Curated Experience  74.73
                  Guindy National Park      National Park  72.82
                      Chetpet Eco Park           Eco Park  71.91
                 Anna Nagar Tower Park          City Park  70.27


In [18]:
results_beach = recommend(
    category="Beach",
    travelling_with="Friends",
    budget="Moderate",
    preferred_time="Evening",
    duration="Half Day",
    interests=["Photography", "Relaxation"],
    top_n=10
)

print(
    results_beach[
        ["place_name", "subcategory", "score"]
    ].to_string(index=False)
)

                         place_name         subcategory  score
      ECR Coastal Sunset Drive Stop  Curated Experience  80.72
                      Akkarai Beach           ECR Beach  79.50
                Thiruvanmiyur Beach Neighbourhood Beach  76.34
                  Injambakkam Beach           ECR Beach  75.91
                  Neelankarai Beach Neighbourhood Beach  75.69
Besant Nagar Beach (Elliot's Beach)         Urban Beach  75.47
                     Panaiyur Beach           ECR Beach  75.12
                    Muttukadu Beach           ECR Beach  75.12
                   VGP Golden Beach           ECR Beach  74.19
                  Kottivakkam Beach Neighbourhood Beach  74.12


In [19]:
if not results.empty:
    top = results.iloc[0]

    print("Place:", top["place_name"])
    print("Category:", top["category"])
    print("Subcategory:", top["subcategory"])
    print("Budget:", top["budget_level"])
    print("Duration:", top["duration_hours"])
    print("Rating:", top["rating_seed"])
    print("Final Score:", top["score"])

Place: Perumbakkam Wetland Park
Category: Nature & Parks
Subcategory: Wetland Park
Budget: Budget
Duration: 2.7
Rating: 4.2
Final Score: 63.84
